# 03 — Complete zone-hour Gold demand

Aggregate all eligible records to zone and hour, construct all 744 January hours for 259 observed zones, record zero-filled assumptions, and save Delta.

**Execution status:** Code is provided for reproducibility. See `results/observed_results.json` for results independently reported from an earlier Fabric run; this generated notebook has not been executed in this environment.

In [ ]:
from pyspark.sql import functions as F
silver=spark.table("silver_yellow_taxi_trips")
eligible=silver.filter(F.col("pickup_eligible_flag")==True)
observed=(eligible.withColumn("pickup_hour",F.date_trunc("hour","tpep_pickup_datetime"))
 .groupBy("PULocationID","pickup_hour").agg(F.count("*").alias("pickup_count")))
print("Observed zone-hours:",observed.count())

In [ ]:
calendar=spark.sql("""SELECT explode(sequence(timestamp'2025-01-01 00:00:00',timestamp'2025-01-31 23:00:00',interval 1 hour)) AS pickup_hour""")
zones=observed.select("PULocationID").distinct()
complete_grid=zones.crossJoin(calendar)
gold=(complete_grid.join(observed,["PULocationID","pickup_hour"],"left")
      .withColumn("pickup_count",F.coalesce(F.col("pickup_count"),F.lit(0))))
print("Zones:",zones.count(),"hours:",calendar.count(),"panel:",gold.count())

In [ ]:
gold_rows=gold.count();zero_rows=gold.filter("pickup_count=0").count();total=gold.agg(F.sum("pickup_count")).first()[0]
dupes=gold.groupBy("PULocationID","pickup_hour").count().filter("count>1").count()
print(dict(rows=gold_rows,zero_rows=zero_rows,total_pickups=total,duplicates=dupes))
assert gold_rows==zones.count()*744
assert total==eligible.count()
assert dupes==0

**Assumption:** Unobserved zone-hour combinations are imputed as 0 recorded TLC Yellow Taxi pickups. This is not proof that passenger demand was 0. The zone universe consists only of zones observed in January.

In [ ]:
TABLE="gold_hourly_zone_demand"
if spark.catalog.tableExists(TABLE):
    print("Existing Gold table unchanged")
else:
    gold.write.format("delta").mode("errorifexists").saveAsTable(TABLE)
assert spark.table(TABLE).count()==gold_rows